### Initialisation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('../outputs/data/train_data.csv')

In [ ]:
df = train_df.copy()

# Select features that matter most (based on diagnostic)
key_features = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1',  # External scores
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',     # Debt ratios
    'DAYS_EMPLOYED_YEARS', 'DAYS_BIRTH_YEARS',         # Tenure & age
    'AMT_CREDIT', 'AMT_INCOME_TOTAL',                  # Loan & income size
    'CNT_FAM_MEMBERS',                                 # Family size
    'EMPLOYMENT_TO_AGE_RATIO',                         # Stability metric
]

# Add any other numeric columns that exist
numeric_cols = df.select_dtypes(include=[np.number]).columns
additional_cols = [c for c in numeric_cols if c not in key_features + ['TARGET', 'SK_ID_CURR']]
key_features.extend(additional_cols[:30])  # Limit to 30 features

# Create feature matrix
X = df[key_features].copy()
y = df['TARGET'].copy()

print(f"Using {X.shape[1]} features")

# Handle missing values
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0)

Using 41 features


### Train-Test Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Default rate in training: {y_train.mean():.2%}")

Training: 246,008 samples
Validation: 61,503 samples
Default rate in training: 8.07%


### Train Optimised Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight='balanced',  # Critical for imbalanced data
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_proba = rf_model.predict_proba(X_val)[:, 1]
y_pred_class = (y_pred_proba > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_val, y_pred_proba)
print(f"\nROC-AUC: {auc:.4f}")


ROC-AUC: 0.7367


### Find Optimal Threshold for Business Use

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

# Find threshold that captures 80% of defaults (high recall)
target_recall = 0.80
closest_idx = (np.abs(recalls - target_recall)).argmin()
optimal_threshold = thresholds[closest_idx] if closest_idx < len(thresholds) else 0.5

print(f"Threshold to capture 80% of defaults: {optimal_threshold:.3f}")

# Apply optimal threshold
y_pred_optimal = (y_pred_proba > optimal_threshold).astype(int)

Threshold to capture 80% of defaults: 0.409


### Segment Analysis with Model Predictions

In [ ]:
# Create results dataframe
results = X_val.copy()
results['actual_default'] = y_val
results['pred_prob'] = y_pred_proba
results['predicted_risky'] = y_pred_optimal

# Segment 1: By credit score
if 'EXT_SOURCE_2' in results.columns:
    low_score = results['EXT_SOURCE_2'] < 0.3
    high_score = results['EXT_SOURCE_2'] > 0.6
    
    print(f"\nEXT_SOURCE_2 segments:")
    print(f"  Low score (<0.3): Model predicted {results[low_score]['predicted_risky'].mean():.1%} as risky")
    print(f"  High score (>0.6): Model predicted {results[high_score]['predicted_risky'].mean():.1%} as risky")

# Segment 2: By debt ratio
if 'CREDIT_INCOME_RATIO' in results.columns:
    high_debt = results['CREDIT_INCOME_RATIO'] > 3.5
    print(f"\nHigh debt (>3.5x): Model flagged {results[high_debt]['predicted_risky'].mean():.1%} as risky")


EXT_SOURCE_2 segments:
  Low score (<0.3): Model predicted 84.9% as risky
  High score (>0.6): Model predicted 26.8% as risky

High debt (>3.5x): Model flagged 48.3% as risky


### Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
for i, row in feature_importance.head(10).iterrows():
    print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")


Top 10 Most Important Features:
  1. EXT_SOURCE_2: 0.2312
  2. EXT_SOURCE_3: 0.2302
  3. EXT_SOURCE_1: 0.0962
  6. DAYS_EMPLOYED_YEARS: 0.0434
  7. DAYS_BIRTH_YEARS: 0.0406
  11. EMPLOYMENT_TO_AGE_RATIO: 0.0344
  16. DAYS_BIRTH: 0.0332
  14. AMT_GOODS_PRICE: 0.0301
  8. AMT_CREDIT: 0.0250
  17. DAYS_EMPLOYED: 0.0249


### Save Results

In [ ]:
import joblib

joblib.dump(rf_model, '../outputs/models/credit_risk_model.pkl')
print("\n> Model saved to outputs/models/credit_risk_model.pkl")

# Save feature importance for dashboard
feature_importance.to_csv('../outputs/results/rf_feature_importance.csv', index=False)
print("> Feature importance saved")

# Save segment analysis for dashboard
segment_data = pd.DataFrame({
    'Segment': ['Low Credit Score (<0.3)', 'High Credit Score (>0.6)',
                'Short Employment (<1 year)', 'Long Employment (>5 years)',
                'Medium DTI (2-3.5x)', 'Low DTI (≤2x)', 'High DTI (>3.5x)'],
    'Default_Rate': [15.88, 4.56, 10.98, 6.41, 8.81, 7.43, 7.95],
    'Sample_Size': [
        (train_df['EXT_SOURCE_2'] < 0.3).sum(),
        (train_df['EXT_SOURCE_2'] > 0.6).sum(),
        (train_df['DAYS_EMPLOYED_YEARS'] < 1).sum(),
        (train_df['DAYS_EMPLOYED_YEARS'] > 5).sum(),
        90638, 76338, 140535
    ]
})
segment_data.to_csv('../outputs/results/segment_analysis.csv', index=False)
print("> Segment analysis saved")


✓ Model saved to outputs/models/credit_risk_model.pkl
✓ Feature importance saved
✓ Segment analysis saved


### Final Summary

In [ ]:
print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                        │
├─────────────────────────────────────────────────────────────┤
│  ROC-AUC (Random Forest):     {auc:.3f}                         │
│  Baseline comparison:         0.701 → {auc:.3f}                 │
│  Improvement:                 +{(auc - 0.701)*100:.1f} percentage pts           │
├─────────────────────────────────────────────────────────────┤
│                    RISK SEGMENTS                            │
├─────────────────────────────────────────────────────────────┤
│  Low EXT_SOURCE_2 (<0.3):      15.88% default (3.5x risk)   │
│  High EXT_SOURCE_2 (>0.6):      4.56% default (baseline)    │
│  Employed <1 year:             10.98% default (1.7x risk)   │
│  Employed >5 years:             6.41% default               │
│  Medium DTI (2-3.5x):           8.81% default (peak risk)   │
├─────────────────────────────────────────────────────────────┤
│                    TOP FEATURES                             │
├─────────────────────────────────────────────────────────────┤
│  #1: {feature_importance.iloc[0]['feature'][:25]}                                           │
│  #2: {feature_importance.iloc[1]['feature'][:25] if len(feature_importance) > 1 else 'N/A'}                                           │
│  #3: {feature_importance.iloc[2]['feature'][:25] if len(feature_importance) > 2 else 'N/A'}                                           │
└─────────────────────────────────────────────────────────────┘
""")


┌─────────────────────────────────────────────────────────────┐
│                    MODEL PERFORMANCE                        │
├─────────────────────────────────────────────────────────────┤
│  ROC-AUC (Random Forest):     0.737                         │
│  Baseline comparison:         0.701 → 0.737                 │
│  Improvement:                 +3.6 percentage pts           │
├─────────────────────────────────────────────────────────────┤
│                    RISK SEGMENTS                            │
├─────────────────────────────────────────────────────────────┤
│  Low EXT_SOURCE_2 (<0.3):      15.88% default (3.5x risk)   │
│  High EXT_SOURCE_2 (>0.6):      4.56% default (baseline)    │
│  Employed <1 year:             10.98% default (1.7x risk)   │
│  Employed >5 years:             6.41% default               │
│  Medium DTI (2-3.5x):           8.81% default (peak risk)   │
├─────────────────────────────────────────────────────────────┤
│                    TOP FEATURES      